## Dependency note

This notebook needs [UCLCHEM](https://github.com/uclchem/UCLCHEM), which is a
Fortran-backed astrochemistry code with no PyPI wheel. It is **not** installed by
`pip install pycalima`, and it cannot be declared as a pyCALIMA extra because
PEP 508 direct references are rejected in distribution metadata.

Build it from source following the upstream instructions, then run this notebook
in the same environment. See `requirements-dev.txt` for the pinned reference.


# Ice Formation and Sticking Coefficient Evolution on Grain Surfaces

This notebook studies the formation of ice monolayers on interstellar dust grains using a modified version of **UCLCHEM**. 
Specifically, we look at the evolution of:
1. The total fractional abundance of ice species.
2. The number of ice monolayers ($N_{\text{monolayers}}$).
3. The sticking coefficient ($S(t)$), which scales dynamically as:
   $$S(t) = S_{\text{ice}} + (S_{\text{dust}} - S_{\text{ice}}) \times e^{-N_{\text{monolayers}}}$$

We assume grains of **$0.1 \mu\text{m}$ radius ($1.0 \times 10^{-5}\text{ cm}$)** with a pure silicate composition (graphite cross-section = 0).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uclchem
from uclchem.model import Cloud

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 14,
    'lines.linewidth': 2,
})

# Define physical constants from surfacereactions.f90 and constants.f90
AMU = 1.66053892e-24  # g
SURFACE_SITE_DENSITY = 1.5e15  # cm^-2

def compute_ice_properties(df, grain_radius, grain_density, gas_dust_mass_ratio, s_dust, s_ice):
    """
    Computes total ice abundance, number of monolayers, and sticking coefficient
    from a UCLCHEM model output dataframe.
    """
    # Sum up all ice species (surface '#' and bulk '@')
    ice_cols = [col for col in df.columns if col.startswith(('#', '@'))]
    total_ice_abundance = df[ice_cols].sum(axis=1)
    
    # Calculate derived parameters
    gas_dust_density_ratio = (4.0 * np.pi * (grain_radius**3) * grain_density * gas_dust_mass_ratio) / (3.0 * AMU)
    num_sites_per_grain = (grain_radius**2) * SURFACE_SITE_DENSITY * 4.0 * np.pi
    
    # Monolayers
    n_mono = total_ice_abundance * gas_dust_density_ratio / num_sites_per_grain
    
    # Sticking coefficient
    s_t = s_ice + (s_dust - s_ice) * np.exp(-n_mono)
    
    return total_ice_abundance, n_mono, s_t

def run_ice_model(param_overrides=None):
    """
    Runs UCLCHEM Cloud model in external (synchronous) mode with specified parameters.
    """
    params = {
        "initialtemp": 20.0,
        "initialdusttemp": 10.0,
        "initialdens": 1e5,
        "finaldens": 1e5,
        "finaltime": 1.0e6,  # 1 Myr to allow full ice development
        "grain_radius": 1.0e-6,  # 0.1 micron
        "grain_density": 3.3,
        "gas_dust_mass_ratio": 100.0,
        "graphite_cross_section": 0.0,  # Pure silicate grain
        "silicate_cross_section": np.pi * (1.0e-5)**2,  # pi * r_g^2
        "grain_crosssection_per_h": 8.0e-22,
        "s_dust": 1.0,
        "s_ice": 0.1,
        "radfield": 1.0,
        "zeta":1.0,
        "fo":1e-6
    }
    if param_overrides:
        params.update(param_overrides)
        
    model = Cloud(param_dict=params, run_type="external")
    model.run()
    
    df = model.get_dataframes(point=0, joined=True)
    
    total_ice, n_mono, s_t = compute_ice_properties(
        df, 
        params["grain_radius"],
        params["grain_density"],
        params["gas_dust_mass_ratio"],
        params["s_dust"],
        params["s_ice"]
    )
    df['total_ice_abundance'] = total_ice
    df['n_mono'] = n_mono
    df['sticking_coefficient'] = s_t
    
    return df, params

### Part 1: Single Test Case of Ice Formation

We begin by running a baseline physical model to investigate the time evolution of:
1. The total abundance of surface/bulk ice species (mainly $H_2O$, $CO$, $CO_2$).
2. The thickness of the ice mantle in terms of monolayers $N_{\text{mono}}$.
3. The sticking coefficient $S(t)$ decaying from $S_{\text{dust}} = 1.0$ to $S_{\text{ice}} = 0.1$ as the grain becomes covered in ice.

We assume standard cloud conditions:
* Gas density: $n_{\text{H}} = 10^5 \text{ cm}^{-3}$
* Gas temperature: $T_{\text{gas}} = 20\text{ K}$
* Dust temperature: $T_{\text{dust}} = 10\text{ K}$
* Grain radius: $0.1\,\mu\text{m}$ (pure silicate, graphite cross-section = 0)

In [ ]:
# Run the baseline model
df_base, params_base = run_ice_model({
    "initialdens": 1e4,
    "finaldens": 1e4,
    "initialtemp": 100.0,
    "initialdusttemp": 70.0
})

time_years = df_base['Time']  # Already in years

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Abundance of key species and total ice
ax1.plot(time_years, df_base['total_ice_abundance'], label='Total Ice', color='black', linestyle='--')
ax1.plot(time_years, df_base['#H2O'] + df_base['@H2O'], label='H2O Ice (Surf+Bulk)', color='blue')
ax1.plot(time_years, df_base['#CO'] + df_base['@CO'], label='CO Ice (Surf+Bulk)', color='red')
ax1.plot(time_years, df_base['#CO2'] + df_base['@CO2'], label='CO2 Ice (Surf+Bulk)', color='green')

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlim(1.0, params_base['finaltime'])
ax1.set_ylim(1e-12, 2e-4)
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Fractional Abundance')
ax1.set_title('Ice Species Abundance Evolution')
ax1.legend(frameon=True, loc='best')

# Subplot 2: Monolayers and sticking coefficient
ax2_twin = ax2.twinx()

color_mono = 'purple'
color_stick = 'darkorange'

ax2.plot(time_years, df_base['n_mono'], label='Monolayers', color=color_mono)
ax2.set_ylabel('Mantle Thickness (Monolayers)', color=color_mono)
ax2.tick_params(axis='y', labelcolor=color_mono)

ax2_twin.plot(time_years, df_base['sticking_coefficient'], label='Sticking Coeff', color=color_stick)
ax2_twin.set_ylabel('Sticking Coefficient S(t)', color=color_stick)
ax2_twin.tick_params(axis='y', labelcolor=color_stick)
ax2_twin.set_ylim(0.0, 1.1)

ax2.set_xscale('log')
ax2.set_xlim(1.0, params_base['finaltime'])
ax2.set_xlabel('Time (years)')
ax2.set_title('Monolayers and Sticking Coefficient Evolution')

# Add legend from both axes
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='best', frameon=True)

plt.tight_layout()

# Print some final values
print("--- Final Timestep Results (t = 1 Myr) ---")
print(f"Total Ice Abundance: {df_base['total_ice_abundance'].iloc[-1]:.4e}")
print(f"Number of Monolayers: {df_base['n_mono'].iloc[-1]:.4f}")
print(f"Sticking Coefficient: {df_base['sticking_coefficient'].iloc[-1]:.4f}")

### Part 2: Grid Study (Varying Density and Dust Temperature)

Now, we perform a parameter space exploration. We study the behavior of pure silicate grains of $0.1\,\mu\text{m}$ radius (no graphite grains) by modifying:
1. Gas density: $n_{\text{H}} \in \{10^4, 10^5, 10^6\} \text{ cm}^{-3}$
2. Dust temperature: $T_{\text{dust}} \in \{10, 15, 20\} \text{ K}$

All simulations are run with a fixed gas temperature of $T_{\text{gas}} = 20\text{ K}$ and for $1\text{ Myr}$.

We will plot:
* $N_{\text{monolayers}}$ over time for each parameter combination.
* The sticking coefficient $S(t)$ over time for each parameter combination.

In [ ]:
densities = [1e4, 1e5, 1e6]
dust_temps = [10.0, 15.0, 20.0]

# Define styling dictionaries
density_colors = {
    1e4: '#3498db', # soft blue
    1e5: '#e74c3c', # soft red
    1e6: '#2ecc71'  # soft green
}

temp_linestyles = {
    10.0: '-',     # Solid
    15.0: '--',    # Dashed
    20.0: ':'      # Dotted
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

grid_results = {}

for dens in densities:
    for tdust in dust_temps:
        print(f"Running model: n_H = 1e{int(np.log10(dens))} cm^-3, T_dust = {tdust} K...")
        
        df, params = run_ice_model({
            "initialdens": dens,
            "finaldens": dens,
            "initialtemp": 20.0,      # fixed gas temp
            "initialdusttemp": tdust  # varying dust temp
        })
        
        grid_results[(dens, tdust)] = df
        
        time_years = df['Time']  # Already in years
        label = f"$n_H=10^{int(np.log10(dens))}, T_d={int(tdust)}$K"
        color = density_colors[dens]
        ls = temp_linestyles[tdust]
        
        # Plot monolayers
        ax1.plot(time_years, df['n_mono'], label=label, color=color, linestyle=ls)
        
        # Plot sticking coefficient
        ax2.plot(time_years, df['sticking_coefficient'], label=label, color=color, linestyle=ls)

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlim(1.0, params['finaltime'])
ax1.set_ylim(1e-4, 1e3)
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Mantle Thickness (Monolayers)')
ax1.set_title('Ice Monolayer Growth (Varying $n_H$ & $T_{dust}$)')
ax1.legend(frameon=True, loc='upper left', ncol=2, fontsize=9)

ax2.set_xscale('log')
ax2.set_xlim(1.0, params['finaltime'])
ax2.set_ylim(0.0, 1.1)
ax2.set_xlabel('Time (years)')
ax2.set_ylabel('Sticking Coefficient $S(t)$')
ax2.set_title('Sticking Coefficient Evolution (Varying $n_H$ & $T_{dust}$)')
ax2.legend(frameon=True, loc='lower left', ncol=2, fontsize=9)

plt.tight_layout()


### Summary and Key Physical Findings

Based on the numerical models run in this study, we can draw the following physical conclusions regarding ice formation on dust grains:

1. **Monolayer Growth Dynamics**:
   * The growth rate of monolayers ($N_{\text{mono}}$) scales strongly with the gas volume density $n_{\text{H}}$. A higher gas density translates to a higher collision frequency between gas-phase species and dust grains, resulting in rapid ice accumulation. At $n_{\text{H}} = 10^6 \text{ cm}^{-3}$, the ice mantle exceeds 1 monolayer within $10^3$ years and reaches several hundred monolayers by $1\text{ Myr}$.
   * At lower gas densities (e.g., $n_{\text{H}} = 10^4 \text{ cm}^{-3}$), the accretion rate is much slower, and the mantle remains thinner (only a few monolayers or less).

2. **Sticking Coefficient Evolution**:
   * The sticking coefficient $S(t)$ decreases exponentially as a function of the number of monolayers, moving from the initial bare dust value ($S_{\text{dust}} = 1.0$) to the ice-covered value ($S_{\text{ice}} = 0.1$).
   * The transition time scales inversely with gas density. In dense regions ($10^6 \text{ cm}^{-3}$), the surface is quickly coated, and the sticking coefficient drops to $S_{\text{ice}}$ in less than $10^3$ years. In lower-density environments, the bare dust surface is exposed for longer periods.

3. **Dust Temperature Effects**:
   * The dust temperature plays a critical role in the composition and stability of the ice. At $T_{\text{dust}} = 20\text{ K}$, thermal desorption of volatile species (like CO) is active, preventing them from forming stable ice. Consequently, the total ice abundance and monolayer growth are suppressed compared to the $10\text{ K}$ and $15\text{ K}$ cases where CO is fully frozen out.